In [1]:
!pip install gymnasium stable-baselines3

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/953.9 kB ? eta -:--:--
   --------------------- ------------------ 524.3/953.9 kB 1.3 MB/s eta 0:00:01
   -------------------------------- ------- 786.4/953.9 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 953.9/953.9 kB 1.1 MB/s  0:00:01

   ------------- -------------------------- 1/3 [gymnasium]
   ------------- -------------------------- 1/3 [gymnasium]
   ------------- -------------------------- 1/3 [gymnasium]
   ------------- -------------------------- 1/3 [gymnasium]
   ------------- -------------------------- 1/3 [gymnasium]
   ------------- -------------------------- 1/3 [gymnasium]
   ------------- ---------------------

In [85]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

In [4]:
train_df = pd.read_csv(r"C:\Users\lanaa\Downloads\train.csv")
train_df.head()

,episode_id,timestep,npc_health,npc_stamina,player_health,distance_to_player,allies_nearby,enemies_nearby,has_cover,escape_route,...,npc_personality,time_since_seen,previous_action,npc_action,npc_health_ratio,player_health_ration,ammo_state,health_difference,ally_enemy_difference,threat_level
0,1,0,69.14,92.93,83.36,6.48,3,3,0,0,...,DEFENSIVE,0,PATROL,ATTACK,0.6914,0.8336,HIGH,-14.22,0,1
1,1,1,69.14,92.93,83.36,6.48,3,3,0,0,...,DEFENSIVE,0,ATTACK,ATTACK,0.6914,0.8336,HIGH,-14.22,0,1
2,1,2,69.14,92.93,83.36,6.48,3,3,0,0,...,DEFENSIVE,0,ATTACK,ATTACK,0.6914,0.8336,HIGH,-14.22,0,3
3,1,3,69.14,92.93,78.84,6.48,3,3,0,0,...,DEFENSIVE,0,ATTACK,CHASE,0.6914,0.7884,HIGH,-9.70,0,1
4,1,4,69.14,90.38,78.84,3.07,3,3,0,0,...,DEFENSIVE,0,CHASE,ATTACK,0.6914,0.7884,HIGH,-9.70,0,1


In [5]:
ACTION_MAP = {
    0: "Attack",
    1: "Chase",
    2: "Patrol",
    3: "Retreat",
    4: "Search",
    5: "Take_cover"
}
print(ACTION_MAP)

{0: 'Attack', 1: 'Chase', 2: 'Patrol', 3: 'Retreat', 4: 'Search', 5: 'Take_cover'}


In [43]:
# in case we will need reverse mapping
ACTION_TO_ID = {
    action: action_id for action_id, action in ACTION_MAP.items()
}
print(ACTION_TO_ID)

{'Attack': 0, 'Chase': 1, 'Patrol': 2, 'Retreat': 3, 'Search': 4, 'Take_cover': 5}


In [44]:
OBSERVATION_COLUMNS = ['npc_health','npc_stamina','player_health',
    'distance_to_player', 'allies_nearby', 'enemies_nearby', 'has_cover',
     'escape_route', 'player_visible', 'player_attacking', 'npc_ammo', 'time_since_seen']

In [45]:
def get_observation(row):
    observation = row[OBSERVATION_COLUMNS].values.astype(np.float32)
    return observation

In [46]:
sample_observation = get_observation(train_df.iloc[0])
print(sample_observation)
print("Observation shape:", sample_observation.shape)

[69.14 92.93 83.36  6.48  3.    3.    0.    0.    1.    0.   30.    0.  ]
Observation shape: (12,)


In [71]:
# episodes
print("Number of episodes:", train_df['episode_id'].nunique())
print("\nFirst 10 episodes IDs:")
print(train_df['episode_id'].unique()[:10])

print('\nEpisode sizes:')
print(train_df.groupby('episode_id').size().describe())

Number of episodes: 1792

First 10 episodes IDs:
[ 1  2  3  4  5  6  7  8  9 10]

Episode sizes:
count    1792.000000
mean       38.875558
std        11.962935
min         8.000000
25%        29.000000
50%        39.000000
75%        49.000000
max        60.000000
dtype: float64


In [73]:
train_df = train_df.sort_values(['episode_id', 'timestep']).reset_index(drop=True)

In [76]:
print(train_df[['episode_id','timestep', 'npc_action']].head(15))

    episode_id  timestep npc_action
0            1         0     ATTACK
1            1         1     ATTACK
2            1         2     ATTACK
3            1         3      CHASE
4            1         4     ATTACK
5            1         5     ATTACK
6            1         6     ATTACK
7            1         7     SEARCH
8            1         8    RETREAT
9            1         9     SEARCH
10           1        10     PATROL
11           1        11     SEARCH
12           1        12    RETREAT
13           1        13     SEARCH
14           1        14     SEARCH


In [77]:
# Creating environment
class NPCBehaviorEnv(gym.Env):
    def __init__(self, data):
        super().__init__()
        self.data = (
            data.sort_values(['episode_id', 'timestep'])
            .reset_index(drop=True))

        self.episodes = self.data['episode_id'].unique()
        
        # Six possible NPC actions
        self.action_space = spaces.Discrete(6)
        # Twelve observation features
        self.observation_space = spaces.Box(
            low=0,
            high=np.inf,
            shape=(12,),
            dtype=np.float32
        )

        self.current_episode = None
        self.episode_data = None
        self.current_step = 0
    # adding reset
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Selecting random episode
        self.current_episode = self.np_random.choice(self.episodes)

        self.episode_data = self.data[self.data['episode_id'] == self.current_episode].reset_index(drop=True)
        
        self.current_step = 0
        row = self.data.iloc[self.current_step]
        observation = get_observation(row)
        info = {
            'episode_id': self.current_episode
        }

        return observation, info
        
    # Adding step
    def step(self, action):

        row = self.data.iloc[self.current_step]

        reward = calculate_reward(row, action)

        # Move to the next timestep
        self.current_step += 1

        terminated = self.current_step >= len(self.episode_data)
        truncated = False

        if not terminated:
            next_row = self.episode_data.iloc[self.current_step]
            observation = get_observation(next_row)
        else:
            observation = np.zeros(
                self.observation_space.shape,
                dtype=np.float32
            )

        info = {
            'episode_id': self.current_episode
        }

        return observation, reward, terminated, truncated, info

In [78]:
## Creating the environment
env = NPCBehaviorEnv(train_df)

print("Action Space:", env.action_space)
print("Observation Space:", env.observation_space)

Action Space: Discrete(6)
Observation Space: Box(0.0, inf, (12,), float32)


In [80]:
##Test Reset
observation, info = env.reset()

print("Episode ID:", info['episode_id'])
print("\nObservation shape:", observation.shape)
print("Current sstep:", env.current_step)
print("Episode length:", len(env.episode_data))
print("\nInfo:")
print(info)

Episode ID: 2085

Observation shape: (12,)
Current sstep: 0
Episode length: 37

Info:
{'episode_id': np.int64(2085)}


In [81]:
# testing several steps
observation, info = env.reset()

print("Episode:", info['episode_id'])
print()

Episode: 1709



In [82]:
for step in range(5):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    print(
    f"Step {step} | "
    f"Action: {ACTION_MAP[action]:12} | "
    f"Reward: {reward:+.2f} | "
    f"Terminated: {terminated}"
)
    if terminated:
        break

Step 0 | Action: PATROL       | Reward: +0.00 | Terminated: False
Step 1 | Action: ATTACK       | Reward: +3.00 | Terminated: False
Step 2 | Action: TAKE_COVER   | Reward: +4.00 | Terminated: False
Step 3 | Action: TAKE_COVER   | Reward: +0.00 | Terminated: False
Step 4 | Action: TAKE_COVER   | Reward: +0.00 | Terminated: False


---WE CAN ADD ENVIRONMENT CHECKER ---

In [83]:
from stable_baselines3.common.env_checker import check_env

In [84]:
check_env(env, warn=True)

In [69]:
# creating some reward function
def calculate_reward(row, action):
    action_name = ACTION_MAP[action]
    reward = 0.0

    # Extract state
    npc_health = row['npc_health']
    npc_stamina = row['npc_stamina']
    distance = row['distance_to_player']
    allies = row['allies_nearby']
    enemies = row['enemies_nearby']
    has_cover = row['has_cover']
    escape_route = row['escape_route']
    player_visible = row['player_visible']
    player_attacking = row['player_attacking']
    ammo = row['npc_ammo']

    # LOW HEALTH
    if npc_health < 30:
        if action_name == "RETREAT":
            reward += 5
        elif action_name == "TAKE_COVER":
            reward += 4
        elif action_name == "ATTACK":
            rewars -= 4

    # PLAYER ATTACKING 
    if player_attacking == 1:
        if action_name == "TAKE_COVER":
            reward += 4
        elif action_name =="RETREAT":
            reward += 3
        elif action_name == "PATROL":
            reward -= 3

    # PLAYER CLOSE
    if distance < 10 and player_visible == 1:
        if action_name == "ATTACK":
            reward += 3
        elif action_name == "CHASE":
            reward += 1

    # PLAYER FAR AWAY
    if distance > 30:
        if action_name == "CHASE":
            reward += 2
    # PLAYER NOT VISIBLE 
    if player_visible == 0:
        if action_name == "SEARCH":
            reward += 2
        elif action_name == "ATTACK":
            reward -= 2
    # NO AMMO
    if ammo == 0:
        if action_name == "ATTACK":
            reward -= 4
        elif action_name == "RETREAT":
            reward += 2
    # COVER AVAILABLE 
    if has_cover == 1 and player_attacking == 1:
        if action_name == "RETREAT":
            reward += 2
    # ALLIES VS ENENIES
    if allies > enemies:
        if action_name == "ATTACK":
            reward += 2
        elif action_name == "RETREAT":
            reward -= 2
    return reward

In [70]:
# testing a reward function
test_row = train_df.iloc[0]

print(test_row[OBSERVATION_COLUMNS])
print("\nDataset action:", test_row['npc_action'])

npc_health            69.14
npc_stamina           92.93
player_health         83.36
distance_to_player     6.48
allies_nearby             3
enemies_nearby            3
has_cover                 0
escape_route              0
player_visible            1
player_attacking          0
npc_ammo                 30
time_since_seen           0
Name: 0, dtype: object

Dataset action: ATTACK
